# Spectrogram Viewers

Click through each species and see its spectrogram next to the numbers the
pipeline measured from it--a quick way to eyeball whether the detection
actually got it right. Three sections below, one per taxon, each with its
own `%store -r` and its own viewer class.

| Viewer | DataFrame | Produced by |
|---|---|---|
| `CricketViewer` | `cricket_final` | `Processing_Cricket_Spectrograms.ipynb` |
| `KatydidViewer` | `katydid_final_multi` | `Processing_Katydid_Spectrograms_Multiple_Bursts.ipynb` |
| `FrogViewer` | `frog_final_multi` | `Processing_Frog_Spectrograms_Multiple_Bursts.ipynb` |

Run the matching processing notebook first so its DataFrame is loaded via `%store`.

## Shared viewer

In [ ]:
import re
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
from PIL import Image

DISCRETE_SIGNALS_DIR = Path.home() / "Discrete_Signals"

In [ ]:
def ordered_unique_species(df):
    """List of (genus, species) pairs in first-appearance order--used instead of
    df.drop_duplicates so the viewer's browse order matches the scrape order."""
    pairs = []
    seen = set()
    for genus, species in zip(df["Genus"], df["Species"]):
        if (genus, species) not in seen:
            seen.add((genus, species))
            pairs.append((genus, species))
    return pairs

In [ ]:
class SpectrogramViewer:
    """Browse species and their spectrograms side by side with the measured
    parameters. Two button rows: previous/next species, and previous/next
    spectrogram within the current species.

    Subclasses fill in the taxon-specific pieces:

    - image_dir(genus, species) / IMAGE_GLOB -- where the images are
    - ID_PATTERN / ID_COLUMN -- how a filename maps back to a DataFrame row
    - format_parameters(row) -- which numbers to print
    - SEEK_FIRST_FRAME -- whether images may be animated GIFs
    """

    ID_PATTERN = r"_spectrogram_(.+)\.[^.]+$"
    ID_COLUMN = None
    IMAGE_GLOB = "{genus}_{species}_spectrogram*"   # formatted with the current pair
    SEEK_FIRST_FRAME = False

    def __init__(self, df):
        self.df = df.reset_index(drop=True)
        self.species_list = ordered_unique_species(self.df)
        self.species_index = 0
        self.image_index = 0

        self.output = widgets.Output()
        buttons = {
            "prev_species": ("← Previous species", lambda _: self.step_species(-1)),
            "next_species": ("Next species →", lambda _: self.step_species(1)),
            "prev_image": ("← Previous spectrogram", lambda _: self.step_image(-1)),
            "next_image": ("Next spectrogram →", lambda _: self.step_image(1)),
        }
        for name, (label, handler) in buttons.items():
            button = widgets.Button(description=label, layout=widgets.Layout(width="190px"))
            button.on_click(handler)
            setattr(self, name, button)

        display(widgets.HBox([self.prev_species, self.next_species]))
        display(widgets.HBox([self.prev_image, self.next_image]))
        display(self.output)
        self.show()

    # --- taxon-specific hooks -------------------------------------------------

    def image_dir(self, genus, species):
        """Folder holding this species' spectrograms (default: one per species)."""
        return DISCRETE_SIGNALS_DIR / self.TAXON_DIR / f"{genus}_{species}"

    def find_images(self, genus, species):
        pattern = self.IMAGE_GLOB.format(genus=genus, species=species)
        return sorted(self.image_dir(genus, species).glob(pattern))

    def format_parameters(self, row):
        raise NotImplementedError

    # --- navigation --------------------------------------------------------------

    def step_species(self, direction):
        self.species_index = min(max(self.species_index + direction, 0), len(self.species_list) - 1)
        self.image_index = 0            # start each species from its first image
        self.show()

    def step_image(self, direction):
        genus, species = self.species_list[self.species_index]
        image_count = len(self.find_images(genus, species))
        if image_count:
            self.image_index = min(max(self.image_index + direction, 0), image_count - 1)
        self.show()

    # --- row lookup + rendering -------------------------------------------------

    def row_for_image(self, genus, species, image_path):
        """The DataFrame row this image was measured from. Matches on the id
        embedded in the filename; falls back to the species' first row."""
        species_rows = self.df[(self.df["Genus"] == genus) & (self.df["Species"] == species)]
        if species_rows.empty:
            return None
        if self.ID_COLUMN and self.ID_COLUMN in species_rows.columns:
            match = re.search(self.ID_PATTERN, image_path.name)
            file_id = match.group(1) if match else None
            hit = species_rows[species_rows[self.ID_COLUMN].astype(str) == str(file_id)]
            if not hit.empty:
                return hit.iloc[0]
        return species_rows.iloc[0]

    def render_image(self, image_path):
        image = Image.open(image_path)
        if self.SEEK_FIRST_FRAME:
            try:
                image.seek(0)          # animated GIFs: show the first frame only
            except EOFError:
                pass
        figure, axes = plt.subplots(figsize=(12, 4))
        axes.imshow(image, cmap="gray")
        axes.axis("off")
        plt.tight_layout()
        plt.show()
        plt.close(figure)              # close explicitly so figures don't pile up per click

    def show(self):
        with self.output:
            clear_output(wait=True)
            genus, species = self.species_list[self.species_index]
            images = self.find_images(genus, species)
            position = f"Species {self.species_index + 1}/{len(self.species_list)}: {genus} {species}"

            if not images:
                print(position)
                print(f"\nNo spectrogram found under {self.image_dir(genus, species)}")
                return

            self.image_index = min(max(self.image_index, 0), len(images) - 1)
            image_path = images[self.image_index]
            row = self.row_for_image(genus, species, image_path)
            if row is None:
                print(f"No data row for {genus} {species}.")
                return

            print(f"{position}   [spectrogram {self.image_index + 1}/{len(images)}]\n")
            for line in self.format_parameters(row):
                print(f"  {line}")
            self.render_image(image_path)

## Crickets

In [ ]:
%store -r cricket_final

In [ ]:
class CricketViewer(SpectrogramViewer):
    TAXON_DIR = "Crickets"
    IMAGE_GLOB = "{genus}_{species}_spectrogram*"
    ID_PATTERN = r"_spectrogram_(.+)\.[^.]+$"
    ID_COLUMN = "File_ID"
    SEEK_FIRST_FRAME = True             # SINA's cricket spectrograms are GIFs

    def format_parameters(self, row):
        return [
            f"Element length:         {row['Element_Length']}",
            f"Inter-element interval: {row['Inter-Element_Interval']}",
            f"Inter-burst interval:   {row['Inter-Burst_Interval']}",
            f"Elements per burst:     {row['Elements_Per_Burst']}",
            f"Temperature:            {row['Temperature']}",
            f"Location:               {row['Location']}",
        ]

In [ ]:
cricket_viewer = CricketViewer(cricket_final)

## Katydids

In [ ]:
%store -r katydid_final_multi

In [ ]:
class KatydidViewer(SpectrogramViewer):
    TAXON_DIR = "Katydids"
    IMAGE_GLOB = "{genus}_{species}_generated_spectrogram*"
    # Match on Spec_ID, not File_ID: File_ID is the audio source's id (kept for
    # traceability in the CSV); the generated PNG is named after the SINA
    # spectrogram, which is what Spec_ID preserves.
    ID_PATTERN = r"_spectrogram_(.+)\.[^.]+$"
    ID_COLUMN = "Spec_ID"
    SEEK_FIRST_FRAME = True

    def format_parameters(self, row):
        return [
            f"Element length:         {row['Element_Length']}",
            f"Inter-element interval: {row['Inter-Element_Interval']}",
            f"Inter-burst interval:   {row['Inter-Burst_Interval']}",
            f"Elements per burst:     {row['Elements_Per_Burst']} "
            f"(min {row['Min_Elements_Per_Burst']}, max {row['Max_Elements_Per_Burst']})",
            f"Burst pattern:          {row['Burst_Pattern']} "
            f"({row['N_Burst_Types_Detected']} type(s), {row['N_Bursts_This_Type']} of this type)",
            f"Temperature:            {row['Temperature']}",
            f"Location:               {row['Location']}",
        ]

In [ ]:
katydid_viewer = KatydidViewer(katydid_final_multi)

## Frogs

Frog spectrograms are stored flat in `Cropped_Frogs_Specs/` rather than one
folder per species, and there's no temperature/location (the clips come from
standalone audio, not SINA), so `FrogViewer` overrides `image_dir` and prints a
shorter parameter block.

In [ ]:
%store -r frog_final_multi

In [ ]:
class FrogViewer(SpectrogramViewer):
    IMAGE_GLOB = "{genus}_{species}_*_spectrogram.png"
    ID_PATTERN = r"_(\d+)_spectrogram\.[^.]+$"   # File_ID is the clip's audio_num
    ID_COLUMN = "File_ID"
    SEEK_FIRST_FRAME = False

    def image_dir(self, genus, species):
        return DISCRETE_SIGNALS_DIR / "Cropped_Frogs_Specs"   # flat, not per-species

    def format_parameters(self, row):
        return [
            f"Element length:         {row['Element_Length']}",
            f"Inter-element interval: {row['Inter-Element_Interval']}",
            f"Inter-burst interval:   {row['Inter-Burst_Interval']}",
            f"Elements per burst:     {row['Elements_Per_Burst']} "
            f"(min {row['Min_Elements_Per_Burst']}, max {row['Max_Elements_Per_Burst']})",
            f"Burst pattern:          {row['Burst_Pattern']} "
            f"({row['N_Burst_Types_Detected']} type(s), {row['N_Bursts_This_Type']} of this type)",
        ]

In [ ]:
frog_viewer = FrogViewer(frog_final_multi)